In [1]:
import torch

def center(K):
    n = K.size(0)
    H = torch.eye(n, device=K.device) - torch.ones(n, n, device=K.device) / n
    return H @ K @ H

def linear_cka(X, Y):
    K = center(X @ X.T)
    L = center(Y @ Y.T)

    hsic = (K * L).sum()
    return hsic / torch.sqrt((K * K).sum() * (L * L).sum())

In [10]:
id_dict={"MedSAM_zs":"MedSAM_zs_ZGT_masses",
         "MedSAM_lora":"medSAM_LoRA_ZGT_allmasses",
        "MammoFM":"MammoFM_ZGT_allmasses",
        "VersaMammo":"VersaMammo_ZGT_allmasses",
        "nnUNet":"nnUNet_ZGT_allmasses"}

near_ood_dict={"MedSAM_zs":"medSAM_zs_INbreast_masses",
               "MedSAM_lora":"medSAM_LoRA_INbreast_allmasses",
        "MammoFM":"MammoFM_INbreast_allmasses",
        "VersaMammo":"VersaMammo_INbreast_allmasses",
        "nnUNet":"nnUNet_INbreast_allmasses"}

far_ood_dict={"MedSAM_zs":"medSAM_zs_MamaMia_masses",
               "MedSAM_lora":"medSAM_LoRA_MamaMia_masses",
        "MammoFM":"MammoFM_MamaMia_masses",
        "VersaMammo":"VersaMammo_MamaMia_masses",
        "nnUNet":"nnUNet_MamaMia_allmasses"}



In [18]:
from itertools import combinations

all_shifts = []

for shift_name, shift_dict in dicts.items():
    shift_list = []

    for (model_A, feats_A), (model_B, feats_B) in combinations(
        shift_dict.items(), 2
    ):
        z1 = torch.load(feats_A)
        z2 = torch.load(feats_B)

        cka = linear_cka(z1, z2)

        combination_name = f"{model_A}-{model_B}"
        shift_list.append({combination_name: cka.item()})

    all_shifts.append({shift_name: shift_list})
    
all_shifts

[{'id_dict': [{'MedSAM_zs-MedSAM_lora': 0.7721615433692932},
   {'MedSAM_zs-MammoFM': 0.23016151785850525},
   {'MedSAM_zs-VersaMammo': 0.13814306259155273},
   {'MedSAM_zs-nnUNet': 0.24691437184810638},
   {'MedSAM_lora-MammoFM': 0.2275887429714203},
   {'MedSAM_lora-VersaMammo': 0.188945010304451},
   {'MedSAM_lora-nnUNet': 0.23926983773708344},
   {'MammoFM-VersaMammo': 0.128959521651268},
   {'MammoFM-nnUNet': 0.2697325646877289},
   {'VersaMammo-nnUNet': 0.1309588998556137}]},
 {'near_ood_dict': [{'MedSAM_zs-MedSAM_lora': 0.999704897403717},
   {'MedSAM_zs-MammoFM': 0.03892083093523979},
   {'MedSAM_zs-VersaMammo': 0.012432216666638851},
   {'MedSAM_zs-nnUNet': 0.21477007865905762},
   {'MedSAM_lora-MammoFM': 0.03930625692009926},
   {'MedSAM_lora-VersaMammo': 0.013266180641949177},
   {'MedSAM_lora-nnUNet': 0.21572476625442505},
   {'MammoFM-VersaMammo': 0.22136084735393524},
   {'MammoFM-nnUNet': 0.23464326560497284},
   {'VersaMammo-nnUNet': 0.06047336384654045}]},
 {'far_ood_d